In [5]:
import os
import re
import pickle as pkl
import ast
from collections import Counter
from typing import List, Tuple, Any
from tqdm import tqdm
import time

import pandas as pd
import numpy as np
import calibration as cal

import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import brier_score_loss, roc_auc_score

import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')
pal = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [6]:
from llm_unsupervised_conf.metrics import *
from llm_unsupervised_conf.math_utils import stack_embeddings
from llm_unsupervised_conf.calibration import fit_predict_prob_models
from llm_unsupervised_conf.plots import plot_reliability, plot_avg_and_worstcase_by_method, plot_method_comparisons
from llm_unsupervised_conf.utils import load_out_df, maybe_add_verbal_conf_row

In [7]:
def _load_verbal_conf_scores(df_save_path_csv: str, ids: pd.Series, stem_idx: int=0) -> np.ndarray | None:
    """Load <df_save_path_csv[:-4]> + '_verbal_conf.csv' and align to ids by string key."""
    if not df_save_path_csv.endswith(".csv"):
        raise ValueError(f"Expected .csv path, got: {df_save_path_csv}")

    vc_path = df_save_path_csv[:-4] + f"_verbal_conf_stem_{stem_idx}.csv"
    if not os.path.exists(vc_path):
        print(f"[verbal_conf] missing: {vc_path}")
        return None

        

    vc_df = pd.read_csv(vc_path)
    if not {"id", "verbal_confidence"}.issubset(vc_df.columns):
        print(f"[verbal_conf] bad schema in {vc_path} (need id, verbal_confidence)")
        return None

    left = pd.DataFrame({"id": ids.astype(str).to_numpy()})
    right = vc_df[["id", "verbal_confidence"]].copy()
    right["id"] = right["id"].astype(str)

    merged = left.merge(right, on="id", how="left")
    return pd.to_numeric(merged["verbal_confidence"], errors="coerce").to_numpy(dtype=float)


def _impute_mean_clip01(x: np.ndarray, fallback: float = 0.5) -> tuple[np.ndarray, float, int]:
    x = np.asarray(x, dtype=float)
    mask = np.isfinite(x)
    mean_val = float(x[mask].mean()) if mask.any() else float(fallback)
    n_missing = int((~mask).sum())
    x = np.where(mask, x, mean_val)
    x = np.clip(x, 0.0, 1.0)
    return x, mean_val, n_missing


def _maybe_add_verbal_conf_row(rows, df_save_path_csv, test_df, correct, n_bins=12, stem_idx=0, verbose=False):
    
    vc = _load_verbal_conf_scores(df_save_path_csv, test_df["id"], stem_idx=stem_idx)
    if vc is None:
        print("Nothing found", stem_idx)
        return rows, None

    vc, mean_val, n_missing = _impute_mean_clip01(vc, fallback=0.5)
    if n_missing and verbose:
        print(f"[verbal_conf_{stem_idx}] imputed {n_missing}/{len(vc)} with mean={mean_val:.4f}")
    elif verbose:
        print(f"[verbal_conf_{stem_idx}] nothing imputed")

    rows.append([
        f"verbal_conf_{stem_idx}",
        get_ece1(vc, correct, n_bins=n_bins),
        get_ece2(vc, correct, n_bins=n_bins),
        get_mce(vc, correct, n_bins=n_bins),
        get_nll(vc, correct),
        brier_score_loss(correct, vc),
        roc_auc_score(correct, vc),
        float(n_missing/len(vc))
    ])
    # print("[verbal_conf] added row")
    return rows, vc


In [8]:
def run_exp(
    dataset,
    model,
    n=1000,
    k_train=100,
    temp_train=0.7,
    temp_test=0.6,
    prob_models=("ridge_clip",),   # <-- pass a list/tuple of the methods above
    include_verbal_conf=True,
    random_state=42,
    n_bins=12,
    embedding_text="question_response",
    drop_bad_rows=True,
    test_prop=0.6,
    verbose=False,
):
    """
    prob_models controls which embedding->prob models to include.
    Example:
      prob_models=["ridge_clip","ridge_logit","hgb_logit","mlp_logit","isotonic_on_ridge"]
    """

    model_name = model.split("/")[-1]

    out_df, df_save_path = load_out_df(dataset, model_name, n, temp_train, k_train, temp_test, embedding_text, drop_bad_rows)

    if verbose or (random_state == 0):
        print(dataset, model, "no rows", len(out_df))

    # --------------------
    # Split
    # --------------------
    train_df, test_df = train_test_split(
        out_df,
        test_size=test_prop,
        random_state=random_state,
        shuffle=True,
    )

    X_train = stack_embeddings(train_df, "embeddings")
    y_train = train_df["consistency"].astype(np.float32).to_numpy()  # target in [0,1]

    X_test = stack_embeddings(test_df, "embeddings")

    # These are for evaluation (your core metrics are vs correctness)
    con_scores = test_df["consistency"].to_numpy(dtype=float)
    correct = test_df["correct"].to_numpy(dtype=int)

    logprobs = test_df["avg_logprobs"].to_numpy(dtype=float)
    ans_logprobs = test_df["ans_logprobs"].to_numpy(dtype=float)
    lp = np.exp(logprobs)
    alp = np.exp(ans_logprobs)

    if verbose:
        print(f"Results: (accuracy={correct.mean():.4f})")

    def metric_row(name, scores, correct, n_bins):
        return [
            name,
            get_ece1(scores, correct, n_bins=n_bins),
            get_ece2(scores, correct, n_bins=n_bins),
            get_mce(scores, correct, n_bins=n_bins),
            get_nll(scores, correct),
            brier_score_loss(correct, scores),
            roc_auc_score(correct, scores),
            0.0
        ]
    
    
    score_sources = {
        "logprob": lp,
        "ans_logprob": alp,
        "oracle_sc": con_scores,
    }
    
    rows = [metric_row(name, scores, correct, n_bins) for name, scores in score_sources.items()]


    # --------------------
    # Embedding->prob models
    # --------------------
    prob_models = list(prob_models) if prob_models is not None else []
    preds = fit_predict_prob_models(
        X_train=X_train,
        y_train_prob=y_train,
        X_test=X_test,
        methods=prob_models,
        random_state=random_state,
    )

    for method_name, pred_probs in preds.items():
        rows.append(metric_row(method_name, pred_probs, correct, n_bins))

    for i in range(10):
        rows, vc_scores = _maybe_add_verbal_conf_row(
            rows, 
            df_save_path, 
            test_df, 
            correct, 
            stem_idx=i, 
            verbose=(random_state==0)
        )

    # --------------------
    # Pack + display
    # --------------------
    exp_df = pd.DataFrame(rows, columns=["Method", "ECE1", "ECE2", "MCE", "NLL", "Brier", "AUROC", "Imputed"])
    exp_df["Model"] = model
    exp_df["Dataset"] = dataset
    exp_df["Accuracy"] = float(correct.mean())
    exp_df = exp_df[["Model", "Dataset", "Method", "ECE1", "ECE2", "MCE", "NLL", "Brier", "AUROC", "Accuracy", "Imputed"]]

    return exp_df


In [30]:
def run_all_exps(models, datasets, exp_key=None, n_trials=10):

    full_df = []
    for dataset in datasets:
        print(f"\n[status] running {dataset}")
        start_time = time.time()
        for model in models:
            print(f"\n[status] running {model}")
            for seed in range(n_trials):
                try:
                    df = run_exp(
                        dataset=dataset,
                        model=model,
                        random_state=seed,
                        prob_models=["ridge_clip"],
                        include_verbal_conf=True,
                        n_bins=12,
                        test_prop=0.6,
                        embedding_text="question_response"
                    )
                    df["seed"] = seed
                    full_df.append(df)
                except Exception as e:
                    print("-*-"*10)
                    print("[No data]")
                    print(model, dataset)
                    print("-*-"*10)
                    break
        print("--- %s seconds ---" % (time.time() - start_time))
    
    full_df = pd.concat(full_df)
    return full_df


In [31]:
models = [
    "Qwen/Qwen3-1.7B", 
    "Qwen/Qwen3-8B",
    "nvidia/Nemotron-Cascade-8B-Thinking",
    "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
]
datasets = [
    "gsm8k",
    "trivia_qa",
]
exp_key = None
n_trials = 10

df = run_all_exps(models, datasets, exp_key, n_trials)


[status] running gsm8k

[status] running Qwen/Qwen3-1.7B
gsm8k Qwen/Qwen3-1.7B no rows 999
[verbal_conf_0] imputed 5/600 with mean=0.9578
[verbal_conf_1] imputed 6/600 with mean=0.9528
[verbal_conf_2] imputed 6/600 with mean=0.9679
[verbal_conf_3] imputed 6/600 with mean=0.9505
[verbal_conf_4] imputed 6/600 with mean=0.9575
[verbal_conf_5] imputed 4/600 with mean=0.9557
[verbal_conf_6] imputed 1/600 with mean=0.9434
[verbal_conf_7] imputed 7/600 with mean=0.9470
[verbal_conf_8] imputed 4/600 with mean=0.9699
[verbal_conf_9] imputed 9/600 with mean=0.9671

[status] running Qwen/Qwen3-8B
gsm8k Qwen/Qwen3-8B no rows 999
[verbal_conf_0] imputed 2/600 with mean=0.9821
[verbal_conf_1] imputed 1/600 with mean=0.9812
[verbal_conf_2] imputed 3/600 with mean=0.9871
[verbal_conf_3] imputed 2/600 with mean=0.9657
[verbal_conf_4] imputed 2/600 with mean=0.9717
[verbal_conf_5] imputed 2/600 with mean=0.9721
[verbal_conf_6] imputed 2/600 with mean=0.9629
[verbal_conf_7] imputed 3/600 with mean=0.964

In [32]:
show_cols = ["Method", "ECE1", "ECE2", "MCE", "Brier", "AUROC", "Imputed"]

In [33]:
df[show_cols].groupby("Method").mean()

,ECE1,ECE2,MCE,Brier,AUROC,Imputed
Method,,,,,,
ans_logprob,0.246954,0.255947,0.364198,0.249009,0.583990,0.000000
logprob,0.177847,0.195624,0.300373,0.193182,0.665056,0.000000
oracle_sc,0.054269,0.067516,0.150866,0.104766,0.814663,0.000000
ridge_clip,0.057931,0.070205,0.143261,0.124222,0.738432,0.000000
verbal_conf_0,0.203322,0.228116,0.339690,0.220377,0.624029,0.008500
verbal_conf_1,0.205381,0.228681,0.372717,0.214674,0.635575,0.014580
verbal_conf_2,0.208571,0.226842,0.346467,0.218401,0.620849,0.018730
verbal_conf_3,0.201508,0.225642,0.369096,0.217701,0.629820,0.015852
verbal_conf_4,0.203927,0.227709,0.344482,0.216351,0.628665,0.011938


In [34]:
def rank_and_aggregate(df, metrics_to_minimize=None, metrics_to_maximize=None):
    """
    Ranks methods based on metrics, calculates an average rank, 
    and provides a final overall ranking.
    """
    # Create a copy to avoid modifying the original dataframe
    ranked_df = df.copy()
    rank_columns = []

    # Handle metrics where LOWER is better (e.g., ECE, Brier, MCE)
    if metrics_to_minimize:
        for metric in metrics_to_minimize:
            col_name = f"{metric}_rank"
            ranked_df[col_name] = ranked_df[metric].rank(ascending=True)
            rank_columns.append(col_name)

    # Handle metrics where HIGHER is better (e.g., AUROC, Accuracy)
    if metrics_to_maximize:
        for metric in metrics_to_maximize:
            col_name = f"{metric}_rank"
            ranked_df[col_name] = ranked_df[metric].rank(ascending=False)
            rank_columns.append(col_name)

    # Calculate the Average Rank across all specified metrics
    ranked_df['avg_rank'] = ranked_df[rank_columns].mean(axis=1)

    # Produce the Final overall ranking based on that average
    ranked_df['overall_rank'] = ranked_df['avg_rank'].rank(ascending=True)

    return ranked_df

In [35]:
grouped_df = df[show_cols+["Model", "Dataset"]].groupby(["Model", "Dataset", "Method"]).mean().reset_index()
grouped_df

,Model,Dataset,Method,ECE1,ECE2,MCE,Brier,AUROC,Imputed
0,Qwen/Qwen3-1.7B,gsm8k,ans_logprob,0.092234,0.092234,0.092234,0.094158,0.604526,0.000000
1,Qwen/Qwen3-1.7B,gsm8k,logprob,0.048619,0.055099,0.092025,0.083452,0.719062,0.000000
2,Qwen/Qwen3-1.7B,gsm8k,oracle_sc,0.055963,0.075238,0.211642,0.067167,0.810532,0.000000
3,Qwen/Qwen3-1.7B,gsm8k,ridge_clip,0.043387,0.053949,0.123213,0.077296,0.737786,0.000000
4,Qwen/Qwen3-1.7B,gsm8k,verbal_conf_0,0.053351,0.053351,0.053351,0.103370,0.603615,0.010500
...,...,...,...,...,...,...,...,...,...
93,nvidia/Nemotron-Cascade-8B-Thinking,trivia_qa,verbal_conf_5,0.367077,0.383709,0.521090,0.367178,0.690478,0.007667
94,nvidia/Nemotron-Cascade-8B-Thinking,trivia_qa,verbal_conf_6,0.386152,0.396259,0.722717,0.383681,0.638791,0.004667
95,nvidia/Nemotron-Cascade-8B-Thinking,trivia_qa,verbal_conf_7,0.405051,0.411506,0.528321,0.404018,0.592944,0.016333
96,nvidia/Nemotron-Cascade-8B-Thinking,trivia_qa,verbal_conf_8,0.365492,0.390276,0.529876,0.377379,0.668312,0.006000


In [36]:
minimize = ['ECE1', 'ECE2', 'MCE', 'Brier', 'Imputed']
maximize = ['AUROC']

vc_methods = [f"verbal_conf_{i}" for i in range(10)]
vc_df = df[df["Method"].isin(vc_methods)]

final_df = rank_and_aggregate(vc_df[show_cols].groupby(["Method"]).mean(), minimize, maximize)

In [37]:
final_df.sort_values(by="overall_rank")

,ECE1,ECE2,MCE,Brier,AUROC,Imputed,ECE1_rank,ECE2_rank,MCE_rank,Brier_rank,Imputed_rank,AUROC_rank,avg_rank,overall_rank
Method,,,,,,,,,,,,,,
verbal_conf_6,0.200645,0.223706,0.389394,0.211162,0.649214,0.008067,2.0,2.0,10.0,1.0,2.0,1.0,3.000000,1.0
verbal_conf_8,0.195259,0.220761,0.362579,0.213753,0.627610,0.008622,1.0,1.0,6.0,2.0,4.0,7.0,3.500000,2.0
verbal_conf_9,0.204227,0.226920,0.336531,0.216380,0.637842,0.006992,7.0,5.0,1.0,6.0,1.0,3.0,3.833333,3.0
verbal_conf_4,0.203927,0.227709,0.344482,0.216351,0.628665,0.011938,6.0,6.0,3.0,5.0,5.0,6.0,5.166667,4.5
verbal_conf_5,0.201761,0.229590,0.355106,0.215680,0.642069,0.014369,4.0,9.0,5.0,4.0,7.0,2.0,5.166667,4.5
verbal_conf_0,0.203322,0.228116,0.339690,0.220377,0.624029,0.008500,5.0,7.0,2.0,9.0,3.0,8.0,5.666667,6.0
verbal_conf_3,0.201508,0.225642,0.369096,0.217701,0.629820,0.015852,3.0,3.0,8.0,7.0,9.0,5.0,5.833333,7.0
verbal_conf_1,0.205381,0.228681,0.372717,0.214674,0.635575,0.014580,8.0,8.0,9.0,3.0,8.0,4.0,6.666667,8.0
verbal_conf_2,0.208571,0.226842,0.346467,0.218401,0.620849,0.018730,9.0,4.0,4.0,8.0,10.0,9.0,7.333333,9.0


In [38]:
comp_table = final_df.sort_values(by="overall_rank").reset_index()[["Method"]+minimize+maximize+["overall_rank"]]

print(comp_table.to_latex(index=False, float_format="%.3f"))

\begin{tabular}{lrrrrrrr}
\toprule
Method & ECE1 & ECE2 & MCE & Brier & Imputed & AUROC & overall_rank \\
\midrule
verbal_conf_6 & 0.201 & 0.224 & 0.389 & 0.211 & 0.008 & 0.649 & 1.000 \\
verbal_conf_8 & 0.195 & 0.221 & 0.363 & 0.214 & 0.009 & 0.628 & 2.000 \\
verbal_conf_9 & 0.204 & 0.227 & 0.337 & 0.216 & 0.007 & 0.638 & 3.000 \\
verbal_conf_4 & 0.204 & 0.228 & 0.344 & 0.216 & 0.012 & 0.629 & 4.500 \\
verbal_conf_5 & 0.202 & 0.230 & 0.355 & 0.216 & 0.014 & 0.642 & 4.500 \\
verbal_conf_0 & 0.203 & 0.228 & 0.340 & 0.220 & 0.008 & 0.624 & 6.000 \\
verbal_conf_3 & 0.202 & 0.226 & 0.369 & 0.218 & 0.016 & 0.630 & 7.000 \\
verbal_conf_1 & 0.205 & 0.229 & 0.373 & 0.215 & 0.015 & 0.636 & 8.000 \\
verbal_conf_2 & 0.209 & 0.227 & 0.346 & 0.218 & 0.019 & 0.621 & 9.000 \\
verbal_conf_7 & 0.227 & 0.249 & 0.363 & 0.233 & 0.013 & 0.608 & 10.000 \\
\bottomrule
\end{tabular}

